In [12]:
import os
from langchain.chat_models import init_chat_model

model = init_chat_model("groq:qwen/qwen3.8-27b")

print(type(model))

<class 'langchain_groq.chat_models.ChatGroq'>


In [13]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(description="title of the movie")
    year: int = Field(description="year the movie was released")
    director: str = Field(description="director of the movie")
    ratings: float = Field(description="movie rating out of 10")

structured_model = model.with_structured_output(Movie)

In [15]:
response = structured_model.invoke(
    "Tell me about the movie hum apke hai kon"
)

print(response)

title='Hum Aapke Hain Koun..!' year=1994 director='Sooraj Barjatya' ratings=8.7


In [ ]:
3#parsed structure

from pydantic import BaseModel , Field
class Movie(BaseModel):
    title: str= Field(..., description="the title of the movie")
    year: int =Field(...,description="the year in which movie realisex")
    director: str = Field(... , description="director of the movie")
    ratings: float = Field(... ,description="movie rating out of 10")

model_with_structure=model.with_structured_output(Movie , include_raw=True)

response=model_with_structure.invoke("provide details about the movie inception")
response


{'raw': AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '8y5f4wvsr', 'function': {'arguments': '{"director":"Christopher Nolan","ratings":8.8,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 68, 'prompt_tokens': 347, 'total_tokens': 415, 'completion_time': 0.182164273, 'completion_tokens_details': None, 'prompt_time': 0.026130346, 'prompt_tokens_details': None, 'queue_time': 0.067571361, 'total_time': 0.208294619}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_424cb89518', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a11034-28ea-7b20-bed3-c9f56b8f16aa-0', tool_calls=[{'name': 'Movie', 'args': {'director': 'Christopher Nolan', 'ratings': 8.8, 'title': 'Inception', 'year': 2010}, 'id': '8y5f4wvsr', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 347, 'output_tokens': 68, 'tot

In [21]:
##nested structure

class Actor(BaseModel):
    name:str
    role:str

class MovieDetails(BaseModel):
    title:str
    year:int
    cast:list[Actor]
    genres:list[str]
    budget: float | None=Field(None , Description="budget in millions usd")

model_with_structure=model.with_structured_output(MovieDetails)

response=model_with_structure.invoke("""
Provide accurate details about the movie 3 Idiots (2009).

Include the main cast members and their actual roles in the movie.
Do not invent actors or roles.
If you are uncertain about any information, leave it out.
Budget should be in millions of USD.
""")
response

C:\Users\Radhika Shukla\AppData\Local\Temp\ipykernel_27316\684194197.py:12: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'Description'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  budget: float | None=Field(None , Description="budget in millions usd")


MovieDetails(title='3 Idiots', year=2009, cast=[Actor(name='Aamir Khan', role='Rancho'), Actor(name='Madhavan', role='Farhan Qureshi'), Actor(name='Sharman Joshi', role='Raju Rastogi'), Actor(name='Kareena Kapoor', role='Tina'), Actor(name='Boman Irani', role='Viru Sahastrabuddhe (Virus)'), Actor(name='Omi Vaidya', role='Professor Chowdhary'), Actor(name='M. Z. Akhtar', role='Chatur'), Actor(name='PO Nair', role='Professor Nair')], genres=['Comedy', 'Drama'], budget=4.1)

In [22]:
from typing_extensions import TypedDict , Annotated

class MovieDict(TypedDict):
    title:Annotated[str , ... , "the title of the movie"]
    year:Annotated[int, ... , "the year the movie was released"]
    director : Annotated[str , ... , "The director of the movie"]
    rating : Annotated[float , ..., "the movies rating out of 10"]

model_withtypedict=model.with_structured_output(MovieDict)
model_withtypedict.invoke("please provide detials about the movie hum apke hai koun")

{'director': 'Sooraj Barjatya',
 'rating': 8.5,
 'title': 'Hum Aapke Hain Koun..!',
 'year': 1994}

In [23]:
class Actor(TypedDict):
    name:str
    role:str

class MovieDetails(TypedDict):
    title:str
    year:int
    cast:list[Actor]
    genres:list[str]
    budget: float | None=Field(None , description="budget in millions usd")

model_with_structure=model.with_structured_output(MovieDetails)

response=model_with_structure.invoke("""
Provide accurate details about the movie 3 Idiots (2009).

Include the main cast members and their actual roles in the movie.
Do not invent actors or roles.
If you are uncertain about any information, leave it out.
Budget should be in millions of USD.
""")
response

{'budget': 5.5,
 'cast': [{'name': 'Aamir Khan', 'role': 'Rancho / Chatur Ramalingam'},
  {'name': 'Madhavan', 'role': 'Farhan Qureshi'},
  {'name': 'Sharman Joshi', 'role': 'Raju Rastogi'},
  {'name': 'Kareena Kapoor', 'role': 'Pia Saraah'},
  {'name': 'Boman Irani', 'role': 'Dr. Viru Sahastrabuddhe (Virus)'}],
 'genres': ['comedy', 'drama'],
 'title': '3 Idiots',
 'year': 2009}

In [ ]:
model_withtypedict.profile

DATACLASSES